## nb_02_game_silver_test

Unit tests for the cleaning/validation functions defined in `nb_02_game_silver`,
using Python's built-in `unittest` framework and a local Spark session.

The functions under test are loaded with `%run nb_02_game_silver { "RUN_PIPELINE": false }`.
Passing `RUN_PIPELINE = False` as a run parameter means only the imports, config,
and function *definitions* from that notebook execute — the `## Run the pipeline`
cells (which read `bronze.game` and write `silver.game`) are skipped entirely, so
this test notebook never touches real bronze/silver tables.

### Load functions under test (pipeline execution skipped)

In [ ]:
%run nb_02_game_silver { "RUN_PIPELINE": false }

### Imports

In [ ]:
import unittest
from datetime import date
from unittest.mock import MagicMock

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

### Test data helpers

Small helper to build a `bronze.game`-shaped DataFrame from plain Python rows,
using the current `spark` session provided by the Fabric/Synapse runtime.

In [ ]:
RAW_SCHEMA = StructType([
    StructField("game_id", StringType(), True),
    StructField("season", StringType(), True),
    StructField("type", StringType(), True),
    StructField("date_time_GMT", StringType(), True),
])


def make_df(rows: list[tuple]):
    """Build a bronze.game-shaped DataFrame from (game_id, season, type, date_time_GMT) tuples."""
    return spark.createDataFrame(rows, schema=RAW_SCHEMA)

### TestLoadBronzeGame

In [ ]:
class TestLoadBronzeGame(unittest.TestCase):
    """load_bronze_game should select exactly the requested columns from bronze.game."""

    def test_selects_only_requested_columns(self):
        fake_df = make_df([
            ("2016020045", "20162017", "R", "2016-10-14T23:00:00Z"),
        ]).withColumn("extra_col", col("game_id"))

        mock_spark = MagicMock()
        mock_spark.sql.return_value = fake_df

        result = load_bronze_game(mock_spark, columns=cols)

        mock_spark.sql.assert_called_once()
        self.assertEqual(result.columns, cols)
        self.assertNotIn("extra_col", result.columns)

    def test_uses_default_columns_when_not_specified(self):
        fake_df = make_df([
            ("2016020045", "20162017", "R", "2016-10-14T23:00:00Z"),
        ])
        mock_spark = MagicMock()
        mock_spark.sql.return_value = fake_df

        result = load_bronze_game(mock_spark)

        self.assertEqual(result.columns, cols)

### TestRemoveDuplicates

In [ ]:
class TestRemoveDuplicates(unittest.TestCase):
    """remove_duplicates should drop rows that share the same DEDUPE_KEYS."""

    def test_drops_exact_key_duplicates(self):
        df = make_df([
            ("g1", "20162017", "R", "2016-10-14T23:00:00Z"),
            ("g1", "20162017", "R", "2016-10-14T23:00:00Z"),  # duplicate key
            ("g2", "20162017", "R", "2016-10-15T23:00:00Z"),
        ])

        result = remove_duplicates(df, keys=DEDUPE_KEYS)

        self.assertEqual(result.count(), 2)
        self.assertEqual(
            {r.game_id for r in result.select("game_id").collect()},
            {"g1", "g2"},
        )

    def test_no_duplicates_is_a_no_op(self):
        df = make_df([
            ("g1", "20162017", "R", "2016-10-14T23:00:00Z"),
            ("g2", "20162017", "R", "2016-10-15T23:00:00Z"),
        ])

        result = remove_duplicates(df, keys=DEDUPE_KEYS)

        self.assertEqual(result.count(), 2)

### TestValidateNoNulls

In [ ]:
class TestValidateNoNulls(unittest.TestCase):
    """validate_no_nulls should drop any row with a null in a required column."""

    def test_drops_rows_with_null_in_required_column(self):
        df = make_df([
            ("g1", "20162017", "R", "2016-10-14T23:00:00Z"),
            ("g2", None, "R", "2016-10-15T23:00:00Z"),        # null season
            ("g3", "20162017", None, "2016-10-16T23:00:00Z"), # null type
            (None, "20162017", "R", "2016-10-17T23:00:00Z"),  # null game_id
        ])

        result = validate_no_nulls(df, cols)

        self.assertEqual(result.count(), 1)
        self.assertEqual(result.collect()[0].game_id, "g1")

    def test_no_nulls_is_a_no_op(self):
        df = make_df([
            ("g1", "20162017", "R", "2016-10-14T23:00:00Z"),
            ("g2", "20162017", "R", "2016-10-15T23:00:00Z"),
        ])

        result = validate_no_nulls(df, cols)

        self.assertEqual(result.count(), 2)

### TestValidateGameType

In [ ]:
class TestValidateGameType(unittest.TestCase):
    """validate_game_type should keep only rows whose 'type' is in VALID_TYPES."""

    def test_drops_invalid_types(self):
        df = make_df([
            ("g1", "20162017", "R", "2016-10-14T23:00:00Z"),
            ("g2", "20162017", "A", "2016-10-15T23:00:00Z"),
            ("g3", "20162017", "P", "2016-10-16T23:00:00Z"),
            ("g4", "20162017", "X", "2016-10-17T23:00:00Z"),  # invalid
            ("g5", "20162017", None, "2016-10-18T23:00:00Z"), # invalid (null)
        ])

        result = validate_game_type(df, VALID_TYPES)

        self.assertEqual(result.count(), 3)
        self.assertEqual(
            {r.type for r in result.select("type").collect()},
            {"R", "A", "P"},
        )

    def test_all_valid_is_a_no_op(self):
        df = make_df([
            ("g1", "20162017", "R", "2016-10-14T23:00:00Z"),
            ("g2", "20162017", "A", "2016-10-15T23:00:00Z"),
        ])

        result = validate_game_type(df, VALID_TYPES)

        self.assertEqual(result.count(), 2)

### TestValidateAndConvertDate

In [ ]:
class TestValidateAndConvertDate(unittest.TestCase):
    """validate_and_convert_date should:
    1. drop rows where date_time_GMT doesn't parse as a timestamp
    2. drop rows where date_time_GMT falls outside the game's season window
       (season start year Sep 15 -> season end year Sep 30)
    3. cast the surviving date_time_GMT to a date-only column
    """

    def test_keeps_valid_in_season_dates_and_casts_to_date(self):
        df = make_df([
            ("g1", "20162017", "R", "2016-10-14T23:00:00Z"),  # valid, within season
        ])

        result = validate_and_convert_date(df)

        self.assertEqual(result.count(), 1)
        row = result.collect()[0]
        self.assertEqual(row.date_time_GMT, date(2016, 10, 14))

    def test_drops_unparseable_timestamp(self):
        df = make_df([
            ("g1", "20162017", "R", "not-a-date"),
        ])

        result = validate_and_convert_date(df)

        self.assertEqual(result.count(), 0)

    def test_drops_date_before_season_start(self):
        df = make_df([
            # season 2016-2017 starts 2016-09-15; this is before that
            ("g1", "20162017", "R", "2016-08-01T00:00:00Z"),
        ])

        result = validate_and_convert_date(df)

        self.assertEqual(result.count(), 0)

    def test_drops_date_after_season_end(self):
        df = make_df([
            # season 2016-2017 ends 2017-09-30; this is after that
            ("g1", "20162017", "R", "2017-10-15T00:00:00Z"),
        ])

        result = validate_and_convert_date(df)

        self.assertEqual(result.count(), 0)

    def test_drops_helper_columns(self):
        df = make_df([
            ("g1", "20162017", "R", "2016-10-14T23:00:00Z"),
        ])

        result = validate_and_convert_date(df)

        for helper in ("season_start", "season_end", "season_start_date", "season_end_date"):
            self.assertNotIn(helper, result.columns)

### TestWriteSilverTable

In [ ]:
class TestWriteSilverTable(unittest.TestCase):
    """write_silver_table should write via the delta format in overwrite mode
    with schema overwrite enabled, to the given table name."""

    def test_writes_with_expected_options(self):
        mock_df = MagicMock()
        mock_writer = mock_df.write
        mock_writer.format.return_value = mock_writer
        mock_writer.mode.return_value = mock_writer
        mock_writer.option.return_value = mock_writer
        mock_df.count.return_value = 42
        mock_df.columns = ["game_id", "season", "type", "date_time_GMT"]

        write_silver_table(mock_df, table_name="silver.game")

        mock_writer.format.assert_called_once_with("delta")
        mock_writer.mode.assert_called_once_with("overwrite")
        mock_writer.option.assert_called_once_with("overwriteSchema", "true")
        mock_writer.saveAsTable.assert_called_once_with("silver.game")

### Run all tests

In [ ]:
loader = unittest.TestLoader()
suite = unittest.TestSuite()
for test_case in (
    TestLoadBronzeGame,
    TestRemoveDuplicates,
    TestValidateNoNulls,
    TestValidateGameType,
    TestValidateAndConvertDate,
    TestWriteSilverTable,
):
    suite.addTests(loader.loadTestsFromTestCase(test_case))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

assert result.wasSuccessful(), "One or more unit tests failed — see output above."